# OpenCodeReasoning challenge Fine-tuning stage 2 (GRPO)

At this point, we must have a model trained with SFT to now be able to generate commentary and some code for a given question-code pair. In this notebook, we train the model to ensure that the code generated by this model is correct.

## About this notebook:
This notebook is derived from Unsloth's GRPO finetuning notebook for Qwen2.5-3B.
There is no solid link for the original notebook. But the original "notebook" can be accessed by the link found at the bottom of the thread in this page: https://github.com/unslothai/unsloth/issues/1634

### Installation

In [1]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
!pip install --upgrade -qqq uv
try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
except: _numpy = "numpy"; _pil = "pillow"
try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
except: is_t4 = False
_vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
!uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
!uv pip install -qqq {_triton} "huggingface_hub>=0.34.0" "datasets==4.3.0"
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

### Unsloth

Load up `Qwen 2.5 3B Instruct`, and set parameters

In [2]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-02-22 11:04:33.254112: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771758273.540640      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771758273.606597      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771758274.162419      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771758274.162484      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771758274.162490      55 computation_placer.cc:177] computation placer alr

INFO 02-22 11:05:13 [__init__.py:244] Automatically detected platform cuda.
ERROR 02-22 11:05:15 [fa_utils.py:57] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
user_secrets = UserSecretsClient()
HUGGINGFACE_TOKEN = user_secrets.get_secret("HF_TOKEN")

login(token=HUGGINGFACE_TOKEN)

In [4]:
MAX_PROMPT_LENGTH     = 2048
MAX_COMPLETION_LENGTH = 4096
MAX_SEQ_LENGTH        = MAX_PROMPT_LENGTH + MAX_COMPLETION_LENGTH

LORA_RANK = 16 # Larger rank = smarter, but slower. I would prefer 64 but no compute. and small model. and complex problem

BASE_MODEL = "santhosh-m/ocr2-sft-lora-merged-v2"

In [5]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True, # False for LoRA 16bit
    fast_inference = True, # Enable vllm fast inference
    max_lora_rank = LORA_RANK,
    gpu_memory_utilization = 1, # Reduce if out of memory
    token=HUGGINGFACE_TOKEN
)

tokenizer.padding_side = "left"

INFO 02-22 11:05:25 [vllm_utils.py:723] Unsloth: Patching vLLM v1 graph capture
INFO 02-22 11:05:25 [vllm_utils.py:752] Unsloth: Patching vLLM v0 graph capture
==((====))==  Unsloth 2026.2.1: Fast Qwen2 patching. Transformers: 4.56.2. vLLM: 0.9.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Standby mode is enabled. However your setting of `gpu_memory_utilization` will OOM.
Changing `gpu_memory_utilization` to 0.8.
Unsloth: vLLM loading santhosh-m/ocr2-sft-lora-merged-v2 with actual GPU utilization = 79.22%
Unsloth: Your GPU has CUDA compute capability 7.5 with VRAM = 14.56 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 6144. Num Se

`torch_dtype` is deprecated! Use `dtype` instead!


INFO 02-22 11:05:50 [config.py:1472] Using max model len 6144
WARNING 02-22 11:05:50 [arg_utils.py:1735] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 
INFO 02-22 11:05:52 [config.py:2285] Chunked prefill is enabled with max_num_batched_tokens=4096.
Unsloth: vLLM Bitsandbytes config using kwargs = {'load_in_8bit': False, 'load_in_4bit': True, 'bnb_4bit_compute_dtype': 'float16', 'bnb_4bit_quant_storage': 'uint8', 'bnb_4bit_quant_type': 'fp4', 'bnb_4bit_use_double_quant': False, 'llm_int8_enable_fp32_cpu_offload': False, 'llm_int8_has_fp16_weight': False, 'llm_int8_skip_modules': [], 'llm_int8_threshold': 6.0}
INFO 02-22 11:05:53 [llm_engine.py:230] Initializing a V0 LLM engine (v0.9.2) with config: model='santhosh-m/ocr2-sft-lora-merged-v2', speculative_config=None, tokenizer='santhosh-m/ocr2-sft-lora-merged-v2', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, d

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

### Adapters cannot be stacked
>`TypeError: Unsloth: Your model already has LoRA adapters. Your new parameters are different.`

At this point I learnt that it is not possible to stack two LoRA adapters of different configurations. We have two options now:
- We can match this new adapter to have the same configuration as the previous one. Or,
- We can merge the adapter and model of the SFT training and start afresh. That way, instead of `MODEL + SFT_ADAPTER + GRPO_ADAPTER` we would now have `(MERGED_SFT_MODEL) + GRPO_ADAPTER`

Update 2: I am proceeding with Option 2. I am using the merged version found at `santhosh-m/ocr2-sft-lora-merged-v2` for this notebook

The LoRA rank `r` and alpha `lora_alpha` were chosen by trial and error to balance between training speed and learning rate.
`lora_alpha = LORA_RANK` was the starting point used in initial experiments. This was subsequently increased to `2*LORA_RANK` after observing slow learning rate.

In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_RANK, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ], # Remove QKVO if out of memory
    lora_alpha = LORA_RANK * 2,
    use_gradient_checkpointing = "unsloth", # Enable long context finetuning
    random_state = 3407,
)

Unsloth 2026.2.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


# Preparing the Dataset

In the next few cells we deal with the dataset to get started with the Reinforcement Learning.

This block below defines a function `format_example` that processes the dataset. The dataset is formatted to include a system prompt and the user prompt. The user prompt includes both the problem question and the user code. Alongside these, the function now attaches a tokenised version of the prompt and attaches sample inputs and outputs to use for GRPO reward function.

The expected format is now given by the system. Along with the three tags mandated by the document (&lt;CoT&gt;, &lt;proposed_fix&gt;, and &lt;test_validation&gt;), I have included two more for a closer flow as trained in Stage-1.

*Also notice that I took the liberty to slightly change these tags. While the document asks for squared-brackets for "proposed fix" with a space and capitalisation, I chose to revert to a more consistent naming of tags*


Update: After some trial and error, it was found that it is more beneficial to have the system prompt as similar as possible to what was used in the previous stage.

In [7]:
import re
from datasets import load_dataset, Dataset

# Define a system prompt that specifies the desired response format.
# SYSTEM_PROMPT_OLD = """
# You are a Quality Assurance Engineer. Evaluate code formally and rigorously.
# For each task, document your reasoning step by step, identify issues, and provide an objective final assessment in a structured format. 
# Always start <CoT> ... </CoT> tags to document your thoughts while assessing the code.
# Then use <judgement> ... </judgement> to provide verdict on whether the code is right or not. This would either be 'right' or 'wrong'
# Then use <answer> ... </answer> to provide the user with a summary of the assessment.
# Follow that with <proposed_fix> ... <proposed_fix>  to provide a correct python code.
# Finally, use the <test_validation> header and come up with at least 3 assert statements, at least one of which tries to reproduce the bug you found above.
# Respond in the following format:
# <CoT>
# ...
# </CoT>
# <judgement>
# ...
# </judgement>
# <answer>
# ...
# </answer>
# <proposed_fix>
# ...
# </proposed_fix>
# <test_validation>
# ...
# </test_validation>

# =The tags above (<CoT>, <judgement>, <answer>, <proposed_fix>, <test_validation>) are custom XML tags used strictly for structured output parsing. Always output them as raw literal XML tags exactly as shown, with no HTML rendering or markdown formatting. You MUST use all five tags in every response, in the exact order shown, and must NOT use any other tags or delimiters outside of these.
# """

# SYSTEM_PROMPT = """You are a Quality Assurance Engineer. Evaluate code formally and rigorously.
# For each task, document your reasoning step by step, identify issues, and provide an objective final assessment in a structured format.
# You must structure your response using XML tags. Always include them literally and in the exact order as shown. Never put any content outside of these XML tags.

# <CoT>
# Document your thoughts step by step while assessing the code.
# </CoT>

# <judgement>
# Your verdict on whether the code is correct. Either 'right' or 'wrong'.
# </judgement>

# <proposed_fix>
# A correct Python code solution.
# </proposed_fix>

# <test_validation>
# At least 3 assert statements with an input and expected output each, where at least one reproduces the bug identified above.
# </test_validation>

# Rules:
# - Never output any text before <CoT> or after </test_validation>.
# - Do not use markdown or HTML formatting inside or outside the tags.
# - The XML tags are structural delimiters only — treat them as such, not as HTML elements."""


# Using a system prompt from previous notebook in an attempt to follow that closely. # TODO Apply the same for the call from UI (or better backend), and explain this in markdown. Explain thta we are going to only get coding question and code from user as inputs. nothing else. this is the reason.
# Update: Yes this works. But the model sometimes spits out other language code T_T

SYSTEM_PROMPT_FROM_PREVIOUS_NOTEBOOK = """You are a Quality Assurance Engineer. Evaluate code formally and rigorously.
For each task, document your reasoning step by step, identify issues, and provide an objective final assessment in a structured format. Avoid informal language or assumptions; base all conclusions on evidence.
Always start with the <CoT>...</CoT> tags and provide your assessment. Follow that with the tags <propposed_fix>...</proposed_fix> with the correct code running python code. Next within <test_validation>...</test_validation> tags provide 3 assert statements that address the bug"""


def format_example(example):
    user_message = f"""I am trying to solve this coding question and wrote this code, but there is something wrong with it. Can you find what the problem is?
The question:
{example['taco_question']}
My code:
{example['ocr_user_code']}
"""
    prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT_FROM_PREVIOUS_NOTEBOOK},
            {"role": "user", "content": user_message},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    return {
        "prompt": prompt,
        "sample_inputs": example["inputs"],
        "sample_outputs": example["outputs"],
    }

### Download the dataset

In [8]:
dataset = load_dataset('santhosh-m/ocr2-wrong-solns-curated', split='train')

README.md:   0%|          | 0.00/730 [00:00<?, ?B/s]

data/train-00000-of-00004.parquet:   0%|          | 0.00/336M [00:00<?, ?B/s]

data/train-00001-of-00004.parquet:   0%|          | 0.00/149M [00:00<?, ?B/s]

data/train-00002-of-00004.parquet:   0%|          | 0.00/108M [00:00<?, ?B/s]

data/train-00003-of-00004.parquet:   0%|          | 0.00/59.7M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/42.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/13226 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1404 [00:00<?, ? examples/s]

### Update V2: Dataset eased for the model
Update: GRPO was failing because all generations produced by the model were too long in the CoT and ended up not generating any valuable code. Therefore, filtering the dataset so that the initial intended CoT is less than 5000 characters long (Same as done in the SFT notebook)

In [9]:
max_CoT_length = 5000

dataset = dataset.filter(
    lambda x: len(x["ocr_qwq_critique"]) <= max_CoT_length)

print(f"Dataset size after filtering by CoT length: {len(dataset)}")

Filter:   0%|          | 0/13226 [00:00<?, ? examples/s]

Dataset size after filtering by CoT length: 856


In [10]:
dataset

Dataset({
    features: ['taco_question', 'ocr_user_code', 'ocr_qwq_critique', 'taco_soln', 'inputs', 'outputs', 'judgement', 'debug_split', 'debug_index'],
    num_rows: 856
})

### Now apply the format

In [11]:
dataset = dataset.map(format_example)

Map:   0%|          | 0/856 [00:00<?, ? examples/s]

### Update V2:
Encountered tensor size mismatch error. Prompts are likely exceeding 4096 tokens, so Unsloth truncates them, but inconsistently across samples in the same batch. This is probably what's giving me tensors of different sizes, like 4 vs 8.

Fixing this by:
* Tokenizing the prompt of every sample,
* Filtering the dataset to only have prompts that fit within MAX_PROMPT_LENGTH after tokenization
* Adding a left-sided padding (see above)

In [12]:
def get_prompt_length(example):
    return len(tokenizer.encode(example["prompt"]))

dataset = dataset.filter(lambda x: get_prompt_length(x) <= MAX_PROMPT_LENGTH)
print(f"Dataset size after filtering by number of tokens in prompt: {len(dataset)}")

Filter:   0%|          | 0/856 [00:00<?, ? examples/s]

Dataset size after filtering by number of tokens in prompt: 855


Defining Reward Functions for GRPO

This section defines two reward functions that assess the model’s responses during fine-tuning. The rewards are based on format adherence and code correctness.

### Explanation of Reward Functions

1. **format**  
   - This function evaluates whether the generated answer (extracted from the XML response) contains all the tags.  

2. **code_correctness**  
   - This function extracts the code part from the response, runs it against the sample inputs, and compares the outputs with the correct sample outputs. It returns a score based on the number of correct outputs obtained.
   - Running code in kaggle using `exec()` is highly unsafe. So for this step, we create a python file with the generated code and run it with the sample inputs we have

Both the above together make sure that there is a code part, and that the code part is correct. GRPO will not focus on the reasoning or chain-of-thought.

#### About test_validation
Since there is no particular field in the dataset that indicates what the problem with the code is, nor the test cases to which it actually fails, I have decided to allow the model to learn this unsupervised.

<!-- (Also this part has a low weightage, so it is alright, I guess. TODO: think about using a different LLM to generate test cases.) -->

### Function that extracts reponse from a completion object

In [13]:
# Extract text from completion
# Somehow I saw the model spit out outputs in two formats:
# 1. The expected 'completions' dictionary
# 2. Just a huge string
# So I am creating a function that handles both cases. Although I am unsure why this is happening

def extract_text(completion, origin_func_name):
    if isinstance(completion, str):
        return completion
    try:
        return completion[0]["content"]
    except Exception as e:
        print(f"[format_presence] {origin_func_name} FAILED to extract text: {e}, completion was: {repr(completion)[:200]}")
        return None


### Rewards for following specified format

The below two functions `format_presence_reward_func` and `format_strict_reward_func` are used to steer the model to follow the format as instructed by the system prompt.

`format_presence_reward_func` is more forgiving and only cares about whether starting tags are present.

`format_strict_reward_func` is strict and attempts to match the exact structure as required by the system prompt

**Comment:** It was interesting to notice that the model was not following system instruction very easily. It involved some training and closely matching with previous system prompt to see some progress.

In [1]:
import re

def format_presence_reward_func(completions):
    required_opening_tags = [
        r"<CoT>",
        r"<judgement>",
        r"<proposed_fix>",
        r"<test_validation>",
    ]
    
    # DEBUG
    # print(f"\n[format_presence] completions type: {type(completions)}")
    # print(f"[format_presence] len: {len(completions)}")
    # print(f"[format_presence] first element type: {type(completions[0])}")
    # print(f"[format_presence] first element: {repr(completions[0])[:300]}")
    
    rewards = []
    for completion in completions:
        
        text = extract_text(completion, "loose_format_func")
        if text is None:
            rewards.append(0.0)
            continue

        # print(f"[format_presence] extracted text[:200]: {repr(text[:200])}")
        score = sum(1 for p in required_opening_tags if re.search(p, text))
        # print(f"[format_presence] score: {score}/{len(required_opening_tags)}")
        rewards.append(score / len(required_opening_tags))
    return rewards


def format_strict_reward_func(completions):
    """Strict: checks tags are properly opened, closed, in order, no extra HTML junk"""
    required_patterns = [
        r"<CoT>.*?</CoT>",
        r"<judgement>.*?</judgement>",
        r"<proposed_fix>.*?</proposed_fix>",
        r"<test_validation>.*?</test_validation>",
    ]
    # Also penalize if there are unexpected HTML-like tags
    html_tag_pattern = r"<(?!CoT|/CoT|judgement|/judgement|proposed_fix|/proposed_fix|test_validation|/test_validation)[a-zA-Z][^>]*>"
    
    rewards = []
    for completion in completions:

        text = extract_text(completion, 'strict_format_func')
        if text is None:
            rewards.append(0.0)
            continue

        
        format_score = sum(
            1 for p in required_patterns
            if re.search(p, text, re.DOTALL)
        ) / len(required_patterns)
        
        # Penalize unwanted HTML tags
        has_html_junk = bool(re.search(html_tag_pattern, text))
        if has_html_junk:
            format_score *= 0.5  # or set to 0.0 if you want to be harsh
        
        rewards.append(format_score)
    return rewards

# Code execution reward

# Important!

NEVER let kaggle run LLM-generated code with exec(), it will end in infinite loops etc.
Use a sandbox for more safety in this matter

The below cell defines a function to safely run Python code. It works by
- Creating a python file
- running it with an input using `subprocess` function
- extracting output and comparing with expected output found in sample_outputs


In [15]:
import subprocess
import tempfile
import os
import sys
import textwrap

def run_code_safely(code: str, input_data: str, timeout: int = 3) -> tuple[bool, str]:
    tmp_path = None
    try:
        with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as tmp_file:
            tmp_file.write(code)
            tmp_path = tmp_file.name

        result = subprocess.run(
            [sys.executable, tmp_path],
            input=input_data,
            capture_output=True,
            text=True,
            timeout=timeout,
        )

        if result.returncode != 0:
            return False, result.stderr.strip()
        return True, result.stdout.strip()

    except subprocess.TimeoutExpired:
        return False, "Timeout"
    except Exception as e:
        return False, str(e)
    finally:
        if tmp_path and os.path.exists(tmp_path):
            os.remove(tmp_path)


`execution_reward_func` extracts the code part found within &lt;proposed_fix&rt; tag to call the `run_code_safely` function. Based on the number of passed executions per set of samples, a reward is calculated.

In [16]:
import re
import traceback

def execution_reward_func(
    completions: list,
    sample_inputs: list,
    sample_outputs: list,
    **kwargs,
) -> list[float]:
    rewards = []
    for completion, s_inputs, s_outputs in zip(completions, sample_inputs, sample_outputs):
        try:
            
            text = extract_text(completion, 'execute_func')
            if text is None:
                rewards.append(0.0)
                continue

            match = re.search(
                r"<proposed_fix>(.*?)</proposed_fix>",
                text,
                re.DOTALL
            )
            if not match:
                rewards.append(0.0)
                continue
            code = match.group(1).strip()
            passed_all = True
            for inp, expected in zip(s_inputs, s_outputs):  # Now per-completion
                success, output = run_code_safely(code, inp)
                if not success:
                    passed_all = False
                    break
                if output.split() != expected.strip().split():
                    passed_all = False
                    break
            rewards.append(1.0 if passed_all else 0.0)
        except Exception:
            rewards.append(0.0)
    return rewards



In [17]:
# debug_code = """

# x = int(input())

# print(x+3)

# """

# inp = '7\n'
# run_code_safely(debug_code, inp)

# Now combine the rewards

~I am giving a weightage of 30% for the format, and 70% for code correctness~

**Update:**
For now I am giving a higher reward for following structure (50%)
But this is just to get the model started. If this training script were able to run for longer on a bigger model, then we may use the intended weightages of (0.15, 0.15, 0.7) for (loose tag structure, strict tag structure, and code correctness), respectively.
Unforunately, because of the complicated nature of this task, it takes more than 10 minutes per training step. Therefore I would focus on structure at this stage.

In [18]:
def combined_reward_func(completions, prompts, **kwargs):
    sample_inputs = kwargs["sample_inputs"]
    sample_outputs = kwargs["sample_outputs"]
    
    presence_scores = format_presence_reward_func(completions)
    strict_scores   = format_strict_reward_func(completions)
    exec_scores     = execution_reward_func(completions, sample_inputs, sample_outputs)
    
    return [
        0.25 * p + 0.25 * s + 0.5 * e
        for p, s, e in zip(presence_scores, strict_scores, exec_scores)
    ]

<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations!
Since this setup takes too much time to run and Kaggle only allows using a single GPU, I am limiting this training to only 25 steps.

Update: It takes longer than 10 minutes per step, therefore I am limiting it down to 20 steps for now.

In [19]:
# shuffle dataset before training

dataset = dataset.shuffle(seed=42)

### Repetition problem:

Sometimes, the model repeats a line of code or an assignment operation infinitely. On inspection, I noticed that a few samples which were used for training in the previous step had a lotof repetitions in the CoT, which may be where our model may have inherited this trait from.

To tackle this, I introduce a `repetition_penalty` parameter. The value of `12` was chosen after trial and error. A smaller penalty allowed for more repetition. But a larger penalty restricted the model from using tag symbols and sometimes restricted the model from completing its statements, further leading to hallucinations.

In [20]:
from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    use_vllm = False, # use vllm for fast inference!
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 1, # Increase to 4 for smoother training
    num_generations = 4, # Decrease if out of memory
    max_prompt_length = MAX_PROMPT_LENGTH,
    max_completion_length = MAX_COMPLETION_LENGTH,
    # num_train_epochs = 1, # Set to 1 for a full training run
    max_steps = 20,                                     # <--------------- at lora_rank 16, 18 steps took an hour.
    save_steps = 250,
    max_grad_norm = 0.1,
    report_to = "none", # Can use Weights & Biases
    output_dir = "outputs",
    repetition_penalty = 1.2
)

In [21]:
# Debug
# print(dataset[0])
# print(dataset.column_names)
# print(dataset[0]['sample_inputs'])
# print(dataset[0]['sample_outputs'])
# print(type(dataset[0]['sample_inputs']))

**Sanity Check 1** ensure correct number of inputs and outputs to avoid tensor dimension error

In [22]:
# test_completions = [[{"role": "assistant", "content": "<CoT>...</CoT><judgement>...</judgement><proposed_fix>...</proposed_fix><test_validation>...</test_validation>"}]] * 4
# test_inputs = dataset[:4]['sample_inputs']
# test_outputs = dataset[:4]['sample_outputs']

# f_scores = format_presence_reward_func(test_completions)
# e_scores = execution_reward_func(test_completions, test_inputs, test_outputs)

# print(f_scores)  # Should be 4
# print(e_scores)  # Should be 4

**Sanity Check 2**
Print a response for one of the data points as a second sanity check

In [23]:
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

inputs = tokenizer(dataset[20]['prompt'], return_tensors="pt").to("cuda")
# outputs = model.generate(**inputs, max_new_tokens=max_tokens_to_use)
outputs = model.generate(**inputs, max_new_tokens=MAX_COMPLETION_LENGTH, repetition_penalty=1.3, temperature=0.7, do_sample=True,)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


system
You are a Quality Assurance Engineer. Evaluate code formally and rigorously.
For each task, document your reasoning step by step, identify issues, and provide an objective final assessment in a structured format. Avoid informal language or assumptions; base all conclusions on evidence.
Always start with the <CoT>...</CoT> tags and provide your assessment. Follow that with the tags <propposed_fix>...</proposed_fix> with the correct code running python code. Next within <test_validation>...</test_validation> tags provide 3 assert statements that address the bug
user
I am trying to solve this coding question and wrote this code, but there is something wrong with it. Can you find what the problem is?
The question:
Can you imagine our life if we removed all zeros from it? For sure we will have many problems.

In this problem we will have a simple example if we removed all zeros from our life, it's the addition operation. Let's assume you are given this equation a + b = c, where a and

**An interesting observation**
The base model can get very unstable!
For the above cell, I have seen the model switch to Chinese at times, showing some evidence that the model may have been trained in a multilingual setting. In other (frequent) cases, the model starts using HTML tags, and these tags are from a coding website. I have seen &lt;submit&lt; tags with "submit solution" written on them, etc. This suggests the model might have scraped through websites like LeetCode and collected coding questions from there. Very interesting stuff.

This behaviour shows that the model may have strong retention of its pretrained patterns and sticks closely to its inherent training rather than its immediate instructions. Note that even though the system instruction explicitly asks the model to not use any HTML tags, it still does.

# Start training

In [24]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        combined_reward_func
    ],
    args = training_args,
    train_dataset = dataset,
)
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 855 | Num Epochs = 1 | Total steps = 20
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 1 x 1) = 4
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / combined_reward_func / mean,rewards / combined_reward_func / std
1,0.000000,0.250000,0.025516,4096.000000,4096.000000,4096.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.250000,0.025516
2,0.000000,0.226562,0.117966,4096.000000,4096.000000,4096.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.226562,0.117966
3,0.000000,0.203125,0.054127,4096.000000,4096.000000,4096.000000,1.000000,0.000000,0.000000,0.000000,0.000016,0.203125,0.054127
4,0.000000,0.187500,0.098821,4096.000000,4096.000000,4096.000000,1.000000,0.000000,0.000000,0.000000,0.000016,0.187500,0.098821
5,0.177900,0.218750,0.098821,2674.500000,1842.000000,4096.000000,0.250000,2200.666748,1842.000000,2680.000000,0.000017,0.218750,0.098821
6,0.136200,0.250000,0.116927,3632.750000,2243.000000,4096.000000,0.750000,2243.000000,2243.000000,2243.000000,0.000020,0.250000,0.116927
7,0.000000,0.296875,0.040344,4096.000000,4096.000000,4096.000000,1.000000,0.000000,0.000000,0.000000,0.000022,0.296875,0.040344
8,0.000000,0.171875,0.093750,4096.000000,4096.000000,4096.000000,1.000000,0.000000,0.000000,0.000000,0.000028,0.171875,0.093750
9,0.000000,0.250000,0.000000,4096.000000,4096.000000,4096.000000,1.000000,0.000000,0.000000,0.000000,0.000029,0.250000,0.000000
10,0.000000,0.226562,0.093315,4096.000000,4096.000000,4096.000000,1.000000,0.000000,0.000000,0.000000,0.000034,0.226562,0.093315


TrainOutput(global_step=20, training_loss=0.06617702266594758, metrics={'train_runtime': 11787.2092, 'train_samples_per_second': 0.007, 'train_steps_per_second': 0.002, 'total_flos': 0.0, 'train_loss': 0.06617702266594758})

### Upload to Hf Hub
Training completed, now I am pushing the model to huggingface hub



In [25]:
from huggingface_hub import login
model.push_to_hub("santhosh-m/grpo_saved_lora-v2", token = HUGGINGFACE_TOKEN) # Online saving
tokenizer.push_to_hub("santhosh-m/grpo-saved-lora-v2", token = HUGGINGFACE_TOKEN) # Online saving

README.md:   0%|          | 0.00/560 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/santhosh-m/grpo_saved_lora-v2


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

<a name="Inference"></a>
### Inference
Now let's try the model we just trained! First, let's first try the model without the GRPO training:

Note: This is **not** the original qwen2.5-coder model; it is the model obtained from SFT Fine-Tuning in the previous step

In [26]:
dataset[20]['prompt']

'<|im_start|>system\nYou are a Quality Assurance Engineer. Evaluate code formally and rigorously.\nFor each task, document your reasoning step by step, identify issues, and provide an objective final assessment in a structured format. Avoid informal language or assumptions; base all conclusions on evidence.\nAlways start with the <CoT>...</CoT> tags and provide your assessment. Follow that with the tags <propposed_fix>...</proposed_fix> with the correct code running python code. Next within <test_validation>...</test_validation> tags provide 3 assert statements that address the bug<|im_end|>\n<|im_start|>user\nI am trying to solve this coding question and wrote this code, but there is something wrong with it. Can you find what the problem is?\nThe question:\nCan you imagine our life if we removed all zeros from it? For sure we will have many problems.\n\nIn this problem we will have a simple example if we removed all zeros from our life, it\'s the addition operation. Let\'s assume you 

In [28]:
text = dataset[20]['prompt']

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = MAX_COMPLETION_LENGTH,
)
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

'<CoT>\nOkay, let\'s see. The problem is to check if after removing all zeros from a, b, and c, the equation a + b = c still holds. Hmm, right. The solution provided takes the input numbers, removes zeros from each, and then checks if the sum of the non-zero a and b equals the non-zero c. \n\nWait, but the code as written might have a problem. Let me think. The input is given in two lines, but the code uses map(int, input().split()). Oh right, that\'s a mistake. Because if the first input is a and the second is b, then the code is splitting into a, b, c. But the input has only two lines. So this code would read three numbers, which is wrong. Oh no, that\'s a bug. Because the code is expecting three numbers, but the input has only two lines. So this is going to cause an error. \n\nSo the code is incorrect because it\'s not processing the input correctly. The first line of input is a, second is b. But the code is taking three numbers from input().split(), which would throw an error. So t

And now with the LoRA we just trained with GRPO - we first save the LoRA locally first so we can use it for inference.

Now we load the LoRA and test:

In [31]:
model.save_lora("grpo_saved_lora-v2")

In [33]:
text = dataset[20]['prompt']

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = MAX_COMPLETION_LENGTH,
    repetition_penalty=1.2
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora-v2"),
)[0].outputs[0].text

output

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/2 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

'<CoT>\nOkay, I need to check if the solution provided correctly handles the problem. Let me think through this step by step.\n\nThe problem states that we have a + b = c, where a and b are positive integers, and the c is their sum. We need to determine whether after removing all zeros from a, b, and c, the resulting numbers form a correct equation.\n\nLooking at the code: \n\nThe function remove_zeros takes an integer n, converts it to a string, replaces all zeros with nothing, then converts back to integer. Then they compute three variables: a_no_zero, b_no_zero, c_no_zero by applying this function to a, b, and c respectively. Wait, wait! But in the code, the variables a, b, c are split into a, b, c here. Oh right, because the input is two lines, each containing an integer. So the code reads a, b, c directly from input as integers, without using map. That\'s important.\n\nWait, the code given here might have a mistake here. Because the input is read in the order a, then b, then c. Wa

Our reasoning model is getting better. It is not particularly correct just yet and will definitely need a lot more steps to truly show its progress and potential. This extends from a few factors:

- We only trained it for 20 steps
- We used a very small model for a very complex task
- The dataset is quite large and can be messy in terms of question structure consistency and commentary accuracy

Unfortunately, the infinite repetition problem still persists, and the full potential of this training pipeline for the use-case of coding through bigger models is yet to be explored.


# Upload final merged version
Finally we may now merge the LoRA Adapter with the model and push this model to HuggingFace Hub



In [34]:
model.push_to_hub_merged("santhosh-m/ocr2-grpo-lora-merged-v2", tokenizer, save_method = "merged_16bit", token = HUGGINGFACE_TOKEN)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...



Unsloth: Copying 1 files from cache to `santhosh-m/ocr2-grpo-lora-merged-v2`:   0%|          | 0/1 [00:00<?, ?it/s]
Unsloth: Copying 1 files from cache to `santhosh-m/ocr2-grpo-lora-merged-v2`: 100%|██████████| 1/1 [00:04<00:00,  4.38s/it]


Successfully copied all 1 files from cache to `santhosh-m/ocr2-grpo-lora-merged-v2`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:00<00:00, 6288.31it/s]

Unsloth: Merging weights into 16bit:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:58<00:00, 58.93s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/santhosh-m/ocr2-grpo-lora-merged-v2`


<a name="Save"></a>
### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("qwen_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("HF_USERNAME/qwen_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False: model.save_pretrained_merged("qwen_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("HF_USERNAME/qwen_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("qwen_lora")
    tokenizer.save_pretrained("qwen_lora")
if False:
    model.push_to_hub("HF_USERNAME/qwen_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/qwen_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("qwen_finetune", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/qwen_finetune", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN",
    )

Now, use the `qwen_finetune.Q8_0.gguf` file or `qwen_finetune.Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).